# 03. MCP Tool Server/Client와 외부 콘텐츠 파이프라인

이 notebook은 원본 `mcp_server.py`, `main.py`, `main2.py`와 Final MCP Project notebook의 흐름을 정리합니다.

- MCP Server: `youtube_trend`, `youtube_music`, `youtube_artist_search`, `calculator` 등록
- MCP Client: stdio 기반 Server 실행, Tool 목록 조회, Tool 호출
- Gemma Tool 계획: 학생 맥락을 보고 JSON 형식의 Tool 호출 계획을 생성하는 실험
- 최종 메시지: Tool 결과의 콘텐츠 제목과 숙제 상태를 개선 프롬프트에 반영

In [ ]:
# %pip install -q -r ../requirements.txt

import os
import sys
import json
import asyncio
from pathlib import Path

CODE_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from dotenv import load_dotenv
load_dotenv(CODE_DIR / ".env")

## 1. MCP Server에 등록된 Tool 확인

In [ ]:
from mcp.main_tool_client import call_tool

tools, trend_result = await call_tool("youtube_trend", {"max_results": 5})
print(json.dumps(tools, ensure_ascii=False, indent=2))
print(json.dumps(trend_result, ensure_ascii=False, indent=2))

## 2. 인기 음악 영상 Tool 호출

원본 `youtube_music_tool.py`는 인기 급상승 영상의 제목에서 음악 관련 키워드를 기준으로 필터링했습니다.

In [ ]:
_, music_result = await call_tool("youtube_music", {"max_results": 5})
print(json.dumps(music_result, ensure_ascii=False, indent=2))

## 3. 관심 가수 기반 검색과 다중 관심사 결과 통합

Final MCP Project에서는 사용자가 입력한 여러 관심 가수를 대상으로 검색 결과를 합쳤습니다.

In [ ]:
_, artist_result = await call_tool(
    "youtube_artist_search",
    {"artists": ["아이유", "에스파"], "max_results": 5},
)
print(json.dumps(artist_result, ensure_ascii=False, indent=2))

## 4. Tool 결과를 최종 동기부여 메시지 프롬프트에 반영

In [ ]:
from src.prompt_templates import build_baseline_prompt, build_improved_prompt

    videos = trend_result.get("videos", [])
    selected_title = videos[0]["title"] if videos else "예시 인기 콘텐츠"

    baseline_prompt = build_baseline_prompt(
        homework_status="완료",
        attendance_summary="이번 학습 챌린지 5회차 인증 (누적 3회 인증)",
        tool_result=selected_title,
    )
    improved_prompt = build_improved_prompt(
        homework_status="완료",
        attendance_summary="이번 학습 챌린지 5회차 인증 (누적 3회 인증)",
        tool_result=selected_title,
        interests=["아이유", "에스파"],
    )

    print("[초기 프롬프트]
", baseline_prompt)
    print("
[개선 프롬프트]
", improved_prompt)

## 5. Gemma Tool 호출 계획 JSON 생성

원본 `main.py`는 Gemma가 Tool 호출 계획을 JSON 형식으로 생성한 뒤, 해당 Tool을 호출하도록 구성했습니다. 실제 모델 로드는 GPU와 Gemma 접근 권한이 필요하므로 아래 셀은 프롬프트만 확인합니다.

In [ ]:
from src.prompt_templates import build_tool_planning_prompt

example_context = {
    "age": 15,
    "lecture": "중학교 2학년 수학",
    "complete": True,
    "interests": ["아이유", "에스파"],
}
print(build_tool_planning_prompt(example_context))